# Pipeline di Analisi Riproducibile per Segnali VLM (S1raw - S7raw) nella Sicurezza Industriale

**Autore:** Data Analytics & Computer Vision Safety Team  
**Scopo:** Analisi del consenso inter-run, attivazione locale e assenza globale dei segnali temporali raw generati da Vision-Language Models (VLM).

---

### Obiettivi Metodologici
1. **Analisi Interna e Riproducibile:** Valutare la consistenza, la ripetibilità e le dinamiche temporali delle attivazioni prodotte da 3 run indipendenti del VLM sullo stesso video industriale.
2. **Assenza Totale Osservata (`global_absence`):** Verificare in modo deterministico se un segnale $S_k\text{raw}$ rimane al di sotto della soglia operativa per l'intera durata del video in tutte le run.
3. **Attivazione Locale e Consenso (`single_signal_interval` & `inter_run_consensus`):** Isolare finestre temporali d'interesse e misurare il consenso inter-run (maggioritario $\ge 2/3$ e completo $3/3$).
4. **Proporzionalità Temporale:** Quantificare il peso relativo di ciascun segnale $S_1\text{raw} - S_7\text{raw}$ all'interno di intervalli temporali specifici.
5. **Esportazione Standardizzata:** Generazione automatica di report tabellari Excel a 6 fogli e grafici ad alta risoluzione (PNG $\ge 150$ DPI).

---

### Limiti Metodologici e Assunzioni Esplicite
* **Nessun Ground Truth:** Questa analisi **NON** costituisce una validazione rispetto a etichette ground truth esterne. Non vengono calcolate metriche di classificazione supervisionata (*Accuracy, Precision, Recall, F1-score*).
* **Nessuna Co-attivazione o Logica Booleana Complessa:** Non vengono valutate correlazioni multivariate o logiche AND/OR tra segnali diversi.
* **Nessuna Interpretazione NLP/Semantica:** I testi generati sono puramente descrittivi dei dati numerici e delle soglie applicate.
* **Dinamicità Totale:** Nessuna assunzione su durata fissa del video (es. 36s), durata costante delle finestre (es. 4s) o numero predeterminato di finestre (es. 18). L'inclusione temporale si basa esclusivamente sulle colonne `starttimes` ed `endtimes`.


## 2. Import delle Librerie

Importazione dei moduli standard di Python, manipolazione dati (`pandas`, `numpy`), visualizzazione (`matplotlib`) e gestione fogli di calcolo (`openpyxl`).


In [ ]:
import os
import sys
from pathlib import Path
import re
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Impostazione stile grafici e rendering
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
plt.rcParams['figure.dpi'] = 150

print(f"[{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Librerie caricate con successo.")


## 3. Configurazione File di Input

Specificare i percorsi dei tre file Excel corrispondenti alle 3 run del VLM.
Il foglio atteso in ciascun file è `vlmonlyresults`.


In [ ]:
# Lista dei percorsi dei 3 file Excel di input (relativi o assoluti)
INPUT_FILES = [
    "pipeline_results_vlm_only X Muletto incidente magazzino 1.xlsx",
    "pipeline_results_vlm_only X Muletto incidente magazzino 2.xlsx",
    "pipeline_results_vlm_only X Muletto incidente magazzino 3.xlsx",
]

SHEET_NAME = "vlm_only_results"

# Finestre da ESCLUDERE da tutte le analisi e dal calcolo delle metriche.
# Inserire qui i Window ID da ignorare, ad esempio:
# EXCLUDED_WINDOWS = [17]
# EXCLUDED_WINDOWS = [3, 8]
# Lasciare [] per non escludere alcuna finestra.
EXCLUDED_WINDOWS = [17]


# Colonne minime obbligatorie da verificare
MANDATORY_COLS = [
    "windowid",
    "starttimes",
    "endtimes",
    "windowdurations",
    "S1raw",
    "S2raw",
    "S3raw",
    "S4raw",
    "S5raw",
    "S6raw",
    "S7raw"
]

ALL_SIGNALS = ["S1raw", "S2raw", "S3raw", "S4raw", "S5raw", "S6raw", "S7raw"]


## 4. Configurazione Soglie di Attivazione ($S_n\text{raw}$)

Un segnale $S_k\text{raw}$ è formalmente considerato **attivo** nella finestra $t$ per la run $r$ se:
$$A[t, k, r] = \begin{cases} 1 & \text{se } S_k\text{raw}[t, r] \ge \text{threshold}[k] \\ 0 & \text{altrimenti} \end{cases}$$

Le soglie sono personalizzabili per singolo segnale.


In [ ]:
# Configurazione delle soglie operative per ciascun segnale raw
SN_THRESHOLDS = {
    "S1raw": 0.50,
    "S2raw": 0.50,
    "S3raw": 0.50,
    "S4raw": 0.50,
    "S5raw": 0.50,
    "S6raw": 0.50,
    "S7raw": 0.50,
}


## 5. Configurazione Query Utente (`CONFIGURAZIONE QUERY UTENTE`)

Questa cella definisce l'insieme delle interrogazioni analitiche da eseguire.
È possibile aggiungere, rimuovere o modificare agevolmente le query senza alterare il codice sottostante.

Tipi di query supportati:
- `global_absence`: Verifica assenza globale del segnale in tutte le finestre e run.
- `single_signal_interval`: Analisi locale di attivazione, consenso e proporzionalità su un intervallo `[inizio_s, fine_s]`.
- `inter_run_consensus`: Analisi esplicita e dettagliata del consenso inter-run in un intervallo.


In [ ]:
# ==========================================
# CONFIGURAZIONE QUERY UTENTE
# ==========================================
QUERIES = [
    # ------------------------------------------------------
    # A. Verifica dell'assenza globale in tutte e 3 le run
    # ------------------------------------------------------
    {
        "id": "Q1_S1_assente_globale",
        "type": "global_absence",
        "signal": "S1raw"
    },
    {
        "id": "Q2_S2_assente_globale",
        "type": "global_absence",
        "signal": "S2raw"
    },
    {
        "id": "Q3_S3_assente_globale",
        "type": "global_absence",
        "signal": "S3raw"
    },
    {
        "id": "Q4_S4_assente_globale",
        "type": "global_absence",
        "signal": "S4raw"
    },
    {
        "id": "Q5_S5_assente_globale",
        "type": "global_absence",
        "signal": "S5raw"
    },
    {
        "id": "Q6_S6_assente_globale",
        "type": "global_absence",
        "signal": "S6raw"
    },
    {
        "id": "Q7_S7_assente_globale",
        "type": "global_absence",
        "signal": "S7raw"
    },

    # ------------------------------------------------------
    # B. Verifica della presenza nell'intero intervallo 0-35
    # ------------------------------------------------------
    {
        "id": "Q8_S1_presenza_0_36",
        "type": "single_signal_interval",
        "signal": "S1raw",
        "time_range_s": [0, 36],
        "consensus_min_runs": 2
    },
    {
        "id": "Q9_S2_presenza_0_36",
        "type": "single_signal_interval",
        "signal": "S2raw",
        "time_range_s": [0, 36],
        "consensus_min_runs": 2
    },
    {
        "id": "Q10_S3_presenza_0_36",
        "type": "single_signal_interval",
        "signal": "S3raw",
        "time_range_s": [0, 36],
        "consensus_min_runs": 2
    },
    {
        "id": "Q11_S4_presenza_0_36",
        "type": "single_signal_interval",
        "signal": "S4raw",
        "time_range_s": [0, 36],
        "consensus_min_runs": 2
    },
    {
        "id": "Q12_S5_presenza_0_36",
        "type": "single_signal_interval",
        "signal": "S5raw",
        "time_range_s": [0, 36],
        "consensus_min_runs": 2
    },
    {
        "id": "Q13_S6_presenza_0_36",
        "type": "single_signal_interval",
        "signal": "S6raw",
        "time_range_s": [0, 36],
        "consensus_min_runs": 2
    },
    {
        "id": "Q14_S7_presenza_0_36",
        "type": "single_signal_interval",
        "signal": "S7raw",
        "time_range_s": [0, 36],
        "consensus_min_runs": 2
    },

    # ------------------------------------------------------
    # C. Verifiche di presenza in intervalli specifici
    # ------------------------------------------------------

]

## 6. Definizione di Costanti e Cartelle di Output

Creazione deterministica della gerarchia delle directory per l'esportazione di grafici ad alta risoluzione e report Excel.


In [ ]:
# Percorsi di output
OUTPUT_DIR = Path("output_sn_raw_analysis")
PLOTS_DIR = OUTPUT_DIR / "plots"
EXCEL_OUTPUT_PATH = OUTPUT_DIR / "risultati_analisi_sn_raw.xlsx"

# Creazione cartelle se non esistenti
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Mappa colori consistente per i segnali S1raw - S7raw
SIGNAL_COLORS = {
    "S1raw": "#1f77b4",  # Blue
    "S2raw": "#ff7f0e",  # Orange
    "S3raw": "#2ca02c",  # Green
    "S4raw": "#d62728",  # Red
    "S5raw": "#9467bd",  # Purple
    "S6raw": "#8c564b",  # Brown
    "S7raw": "#e377c2"   # Pink
}

RUN_COLORS = {
    1: "#2b5c8f",
    2: "#d95f02",
    3: "#7570b3"
}

print(f"Directory principale output: {OUTPUT_DIR.resolve()}")
print(f"Directory grafici: {PLOTS_DIR.resolve()}")


## 7. Funzioni per Lettura e Validazione File di Input

Verifica l'esistenza fisica dei file, carica il foglio `vlmonlyresults`, valida la presenza e i tipi delle colonne obbligatorie ed effettua la pulizia dei dati numerici.


In [ ]:
def load_and_validate_run_file(file_path: str, sheet_name: str = "vlmonlyresults", mandatory_cols: list = None) -> pd.DataFrame:
    """
    Carica e valida un singolo file Excel di run VLM.
    
    Parametri:
        file_path (str): Percorso del file Excel.
        sheet_name (str): Nome del foglio da caricare.
        mandatory_cols (list): Lista delle colonne obbligatorie.
        
    Ritorna:
        pd.DataFrame: DataFrame validato e ordinato per windowid.
    """
    if mandatory_cols is None:
        mandatory_cols = MANDATORY_COLS
        
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Errore: Il file '{file_path}' non esiste.")
        
    try:
        df = pd.read_excel(path, sheet_name=sheet_name)
    except Exception as e:
        raise ValueError(f"Errore durante la lettura del foglio '{sheet_name}' in '{file_path}': {e}")
        
    # Normalizzazione dei nomi delle colonne.
    # I file VLM-only usano nomi descrittivi con underscore (es. window_id,
    # start_time_s, S1_raw), mentre la pipeline lavora con nomi canonici
    # windowid, starttimes, S1raw, ecc.
    column_aliases = {
        "window_id": "windowid",
        "start_time_s": "starttimes",
        "end_time_s": "endtimes",
        "window_duration_s": "windowdurations",
        "S1_raw": "S1raw",
        "S2_raw": "S2raw",
        "S3_raw": "S3raw",
        "S4_raw": "S4raw",
        "S5_raw": "S5raw",
        "S6_raw": "S6raw",
        "S7_raw": "S7raw",
    }

    rename_map = {col: column_aliases[col] for col in df.columns if col in column_aliases}
    if rename_map:
        df = df.rename(columns=rename_map)

    # Verifica eventuali collisioni dopo la normalizzazione.
    duplicated_cols = df.columns[df.columns.duplicated()].tolist()
    if duplicated_cols:
        raise ValueError(
            f"File '{file_path}' non valido: colonne duplicate dopo la normalizzazione -> {duplicated_cols}"
        )

    # Verifica colonne obbligatorie
    missing_cols = [c for c in mandatory_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"File '{file_path}' non valido: colonne mancanti -> {missing_cols}")
        
    # Conversione robusta a valori numerici
    numeric_cols = ["windowid", "starttimes", "endtimes", "windowdurations"] + [c for c in mandatory_cols if c.startswith("S")]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].isna().any():
                nan_count = df[col].isna().sum()
                raise ValueError(f"File '{file_path}': trovati {nan_count} valori non numerici/NaN nella colonna obbligatoria '{col}'.")
                
    # Verifica unicità windowid
    if df["windowid"].duplicated().any():
        dup_ids = df[df["windowid"].duplicated()]["windowid"].tolist()
        raise ValueError(f"File '{file_path}': windowid duplicati rilevati -> {dup_ids}")
        
    # Ordinamento per windowid
    df["windowid"] = df["windowid"].astype(int)
    df = df.sort_values("windowid").reset_index(drop=True)
    
    return df


## 8. Funzioni per Allineamento delle 3 Run

Verifica la coerenza dei metadati temporali (`windowid`, `starttimes`, `endtimes`, `windowdurations`) tra le 3 run e costruisce una struttura unificata multilivello.


In [ ]:
def align_runs(run_dfs: list) -> tuple:
    """
    Allinea le 3 run su windowid e verifica la coerenza temporale.
    
    Parametri:
        run_dfs (list of pd.DataFrame): Lista dei 3 DataFrame corrispondenti a Run 1, 2, 3.
        
    Ritorna:
        tuple: (df_aligned_temporal, dict_signals_raw)
            - df_aligned_temporal: DataFrame con metadati temporali condivisi
            - dict_signals_raw: dict {signal: pd.DataFrame con colonne ['run1', 'run2', 'run3']}
    """
    if len(run_dfs) != 3:
        raise ValueError(f"Richieste esattamente 3 run, ricevute: {len(run_dfs)}")
        
    # Verifica che i windowid coincidano esattamente tra tutte le run
    ref_windowids = run_dfs[0]["windowid"].values
    for idx, df in enumerate(run_dfs[1:], start=2):
        if not np.array_equal(ref_windowids, df["windowid"].values):
            diff = set(ref_windowids).symmetric_difference(set(df["windowid"].values))
            raise ValueError(f"Disallineamento rilevato tra Run 1 e Run {idx} nei windowid: differenze -> {diff}")
            
    # Base temporale comune (estratta da Run 1, verificata con le altre)
    temporal_cols = ["windowid", "starttimes", "endtimes", "windowdurations"]
    df_temporal = run_dfs[0][temporal_cols].copy()
    
    # Verifica coerenza start/end times
    for idx, df in enumerate(run_dfs[1:], start=2):
        if not np.allclose(df_temporal["starttimes"], df["starttimes"], atol=1e-3):
            print(f"Attenzione: Lievi discrepanze in 'starttimes' tra Run 1 e Run {idx}.")
        if not np.allclose(df_temporal["endtimes"], df["endtimes"], atol=1e-3):
            print(f"Attenzione: Lievi discrepanze in 'endtimes' tra Run 1 e Run {idx}.")

    # Dizionario con matrici dei segnali raw per ciascun S1..S7
    dict_signals_raw = {}
    for sig in ALL_SIGNALS:
        sig_df = pd.DataFrame({
            "windowid": df_temporal["windowid"],
            "starttimes": df_temporal["starttimes"],
            "endtimes": df_temporal["endtimes"],
            "windowdurations": df_temporal["windowdurations"],
            "run1": run_dfs[0][sig].values,
            "run2": run_dfs[1][sig].values,
            "run3": run_dfs[2][sig].values,
        })
        dict_signals_raw[sig] = sig_df
        
    return df_temporal, dict_signals_raw


## 9. Funzioni per Binarizzazione e Calcolo del Consenso

Applicazione della soglia di attivazione per determinare:
* Attivazione binaria per singola run ($0$ o $1$).
* Numero di run attive ($0, 1, 2, 3$).
* Consenso maggioritario (almeno 2 run attive su 3).
* Consenso completo (3 run attive su 3).


In [ ]:
def binarize_and_compute_consensus(sig_df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """
    Aggiunge colonne binarie e metriche di consenso inter-run a un DataFrame di segnale.
    
    Parametri:
        sig_df (pd.DataFrame): DataFrame con colonne ['run1', 'run2', 'run3'].
        threshold (float): Soglia di attivazione.
        
    Ritorna:
        pd.DataFrame: DataFrame arricchito con 'bin_run1', 'bin_run2', 'bin_run3',
                      'active_runs_count', 'consensus_str', 'majority_consensus', 'complete_consensus'.
    """
    df = sig_df.copy()
    df["bin_run1"] = (df["run1"] >= threshold).astype(int)
    df["bin_run2"] = (df["run2"] >= threshold).astype(int)
    df["bin_run3"] = (df["run3"] >= threshold).astype(int)
    
    df["active_runs_count"] = df["bin_run1"] + df["bin_run2"] + df["bin_run3"]
    df["consensus_str"] = df["active_runs_count"].astype(str) + "/3"
    df["majority_consensus"] = df["active_runs_count"] >= 2
    df["complete_consensus"] = df["active_runs_count"] == 3
    
    return df


## 10. Funzioni per Selezione Dinamica delle Finestre Temporali

La selezione delle finestre si basa sull'effettiva sovrapposizione temporale:
$$\max(\text{starttimes}, \text{inizio\_intervallo}) < \min(\text{endtimes}, \text{fine\_intervallo})$$

Questo metodo garantisce la corretta inclusione di finestre complete e parzialmente sovrapposte, senza assumere durate fisse.


In [ ]:
def select_windows_in_interval(df_temporal: pd.DataFrame, time_range_s: list) -> pd.DataFrame:
    """
    Seleziona le finestre temporali che si sovrappongono all'intervallo [inizio_s, fine_s].
    
    Condizione: max(starttimes, t_start) < min(endtimes, t_end)
    
    Parametri:
        df_temporal (pd.DataFrame): DataFrame contenente starttimes e endtimes.
        time_range_s (list or tuple): [t_start, t_end] in secondi.
        
    Ritorna:
        pd.DataFrame: Sottoinsieme delle finestre temporali selezionate.
    """
    if len(time_range_s) != 2:
        raise ValueError(f"time_range_s deve essere una lista di 2 elementi [t_start, t_end], ricevuto: {time_range_s}")
        
    t_start, t_end = float(time_range_s[0]), float(time_range_s[1])
    if t_start >= t_end:
        raise ValueError(f"Inizio intervallo ({t_start}s) deve essere strettamente minore della fine ({t_end}s).")
        
    # Calcolo sovrapposizione
    overlap_condition = np.maximum(df_temporal["starttimes"], t_start) < np.minimum(df_temporal["endtimes"], t_end)
    selected_df = df_temporal[overlap_condition].copy()
    
    if selected_df.empty:
        print(f"Attenzione: Nessuna finestra temporale trovata per l'intervallo [{t_start}s, {t_end}s].")
        
    return selected_df


## 11. Funzioni per Analisi di Assenza Globale (`global_absence`)

Verifica se un segnale $S_k\text{raw}$ rimane sempre al di sotto della soglia in tutte le finestre e su tutte le 3 run.


In [ ]:
def analyze_global_absence(query_cfg: dict, dict_signals_raw: dict, thresholds: dict) -> dict:
    """
    Esegue l'analisi di assenza globale per una specifica query.
    
    Parametri:
        query_cfg (dict): Configurazione della query (id, signal, type).
        dict_signals_raw (dict): Mappa dei segnali raw.
        thresholds (dict): Mappa delle soglie.
        
    Ritorna:
        dict: Risultati aggregati e tabellari dell'analisi.
    """
    sig_name = query_cfg["signal"]
    threshold = thresholds[sig_name]
    
    if sig_name not in dict_signals_raw:
        raise ValueError(f"Segnale '{sig_name}' non trovato nei dati.")
        
    sig_df = dict_signals_raw[sig_name]
    df_bin = binarize_and_compute_consensus(sig_df, threshold)
    
    max_global = float(df_bin[["run1", "run2", "run3"]].max().max())
    max_run1 = float(df_bin["run1"].max())
    max_run2 = float(df_bin["run2"].max())
    max_run3 = float(df_bin["run3"].max())
    
    act_run1 = int(df_bin["bin_run1"].sum())
    act_run2 = int(df_bin["bin_run2"].sum())
    act_run3 = int(df_bin["bin_run3"].sum())
    act_total = act_run1 + act_run2 + act_run3
    
    always_below = (act_total == 0)
    
    # Distribuzione consenso
    c_counts = df_bin["active_runs_count"].value_counts().to_dict()
    consensus_0_3 = int(c_counts.get(0, 0))
    consensus_1_3 = int(c_counts.get(1, 0))
    consensus_2_3 = int(c_counts.get(2, 0))
    consensus_3_3 = int(c_counts.get(3, 0))
    
    if always_below:
        desc_text = f"{sig_name} non è stata attivata in alcuna finestra delle tre run secondo la soglia impostata ({threshold:.2f})."
    else:
        desc_text = f"{sig_name} ha registrato {act_total} attivazioni complessive su 3 run (Max globale: {max_global:.4f}, soglia: {threshold:.2f})."
        
    summary_row = {
        "query_id": query_cfg["id"],
        "signal": sig_name,
        "threshold": threshold,
        "always_below_threshold": "SÌ" if always_below else "NO",
        "max_global": max_global,
        "max_run1": max_run1,
        "max_run2": max_run2,
        "max_run3": max_run3,
        "activations_run1": act_run1,
        "activations_run2": act_run2,
        "activations_run3": act_run3,
        "activations_total": act_total,
        "consensus_0_3_count": consensus_0_3,
        "consensus_1_3_count": consensus_1_3,
        "consensus_2_3_count": consensus_2_3,
        "consensus_3_3_count": consensus_3_3,
        "descriptive_text": desc_text
    }
    
    return {
        "query_id": query_cfg["id"],
        "query_type": "global_absence",
        "signal": sig_name,
        "threshold": threshold,
        "summary": summary_row,
        "window_details": df_bin,
        "always_below": always_below
    }


## 12. Funzioni per Analisi di Intervallo Locale (`single_signal_interval`)

Analizza l'attivazione di un singolo segnale in un intervallo temporale, calcolando percentuali di copertura per consenso maggioritario e completo.


In [ ]:
def analyze_single_signal_interval(query_cfg: dict, dict_signals_raw: dict, thresholds: dict) -> dict:
    """
    Esegue l'analisi di attivazione locale e consenso per una query su intervallo.
    
    Parametri:
        query_cfg (dict): Configurazione della query.
        dict_signals_raw (dict): Mappa dei segnali raw.
        thresholds (dict): Mappa delle soglie.
        
    Ritorna:
        dict: Risultati analitici sull'intervallo temporale.
    """
    sig_name = query_cfg["signal"]
    time_range = query_cfg["time_range_s"]
    threshold = thresholds[sig_name]
    min_runs = query_cfg.get("consensus_min_runs", 2)
    
    sig_df = dict_signals_raw[sig_name]
    df_bin_full = binarize_and_compute_consensus(sig_df, threshold)
    
    # Selezione finestre
    t_start, t_end = float(time_range[0]), float(time_range[1])
    overlap_condition = np.maximum(df_bin_full["starttimes"], t_start) < np.minimum(df_bin_full["endtimes"], t_end)
    df_interval = df_bin_full[overlap_condition].copy().reset_index(drop=True)
    
    n_windows = len(df_interval)
    if n_windows > 0:
        window_ids_str = ", ".join(df_interval["windowid"].astype(str).tolist())
        maj_count = int(df_interval["majority_consensus"].sum())
        comp_count = int(df_interval["complete_consensus"].sum())
        maj_coverage = (maj_count / n_windows) * 100.0
        comp_coverage = (comp_count / n_windows) * 100.0
        
        act_r1 = int(df_interval["bin_run1"].sum())
        act_r2 = int(df_interval["bin_run2"].sum())
        act_r3 = int(df_interval["bin_run3"].sum())
        max_obs = float(df_interval[["run1", "run2", "run3"]].max().max())
        
        desc_text = (f"Nell'intervallo [{t_start}s, {t_end}s] ({n_windows} finestre: ID [{window_ids_str}]), "
                     f"{sig_name} ha consenso maggioritario nel {maj_coverage:.1f}% delle finestre "
                     f"({maj_count}/{n_windows}) e consenso completo nel {comp_coverage:.1f}% ({comp_count}/{n_windows}). "
                     f"Attivazioni per run: R1={act_r1}, R2={act_r2}, R3={act_r3}. Max osservato: {max_obs:.4f}.")
    else:
        window_ids_str = "Nessuna"
        maj_count, comp_count = 0, 0
        maj_coverage, comp_coverage = 0.0, 0.0
        act_r1, act_r2, act_r3 = 0, 0, 0
        max_obs = 0.0
        desc_text = f"Nessuna finestra temporale si sovrappone all'intervallo [{t_start}s, {t_end}s]."
        
    summary_row = {
        "query_id": query_cfg["id"],
        "signal": sig_name,
        "time_start_s": t_start,
        "time_end_s": t_end,
        "time_range_str": f"[{t_start}s, {t_end}s]",
        "included_windows_ids": window_ids_str,
        "num_windows": n_windows,
        "threshold": threshold,
        "consensus_min_runs_required": min_runs,
        "majority_consensus_windows": maj_count,
        "majority_consensus_coverage_pct": maj_coverage,
        "complete_consensus_windows": comp_count,
        "complete_consensus_coverage_pct": comp_coverage,
        "activations_run1": act_r1,
        "activations_run2": act_r2,
        "activations_run3": act_r3,
        "max_observed_value": max_obs,
        "descriptive_notes": desc_text
    }
    
    return {
        "query_id": query_cfg["id"],
        "query_type": "single_signal_interval",
        "signal": sig_name,
        "time_range": time_range,
        "threshold": threshold,
        "summary": summary_row,
        "window_details": df_interval
    }


## 13. Funzioni per Consenso Inter-Run (`inter_run_consensus`)

Fornisce un'analisi di consenso specifica per intervallo per audit e validazione incrociata tra le tre run.


In [ ]:
def analyze_inter_run_consensus(query_cfg: dict, dict_signals_raw: dict, thresholds: dict) -> dict:
    """
    Esegue l'analisi approfondita di consenso inter-run.
    """
    # Sfrutta la logica di single_signal_interval con output arricchito
    return analyze_single_signal_interval(query_cfg, dict_signals_raw, thresholds)


## 14. Funzioni per Calcolo della Proporzionalità dei Segnali

### Definizione Metodologica Applicata
Per ciascun intervallo temporale analizzato, la proporzionalità relativa tra i segnali $S_1\text{raw} - S_7\text{raw}$ è calcolata come:
$$\text{Attivazioni Totali}(S_k) = \sum_{t \in \text{intervallo}} \sum_{r=1}^3 A[t, k, r]$$
$$\text{Percentuale}(S_k) = \frac{\text{Attivazioni Totali}(S_k)}{\sum_{j=1}^7 \text{Attivazioni Totali}(S_j)} \times 100$$

*Nota:* Questa metrica quantifica la presenza complessiva di ciascun segnale su tutte le run nell'intervallo temporale considerato. I segnali con $0$ attivazioni totali vengono isolati e documentati in tabella separata senza distorcere il grafico a torta.


In [ ]:
def compute_interval_proportionality(time_range_s: list, dict_signals_raw: dict, thresholds: dict) -> tuple:
    """
    Calcola il conteggio e la quota percentuale di attivazioni per tutti i segnali S1raw-S7raw nell'intervallo.
    
    Parametri:
        time_range_s (list): [t_start, t_end]
        dict_signals_raw (dict): Mappa dei segnali raw.
        thresholds (dict): Mappa delle soglie.
        
    Ritorna:
        tuple: (df_proportionality, active_only_df, zero_signals_list)
    """
    t_start, t_end = float(time_range_s[0]), float(time_range_s[1])
    
    counts = {}
    counts_run1 = {}
    counts_run2 = {}
    counts_run3 = {}
    
    for sig in ALL_SIGNALS:
        sig_df = dict_signals_raw[sig]
        thresh = thresholds[sig]
        df_bin = binarize_and_compute_consensus(sig_df, thresh)
        
        # Filtro intervallo
        overlap = np.maximum(df_bin["starttimes"], t_start) < np.minimum(df_bin["endtimes"], t_end)
        df_sub = df_bin[overlap]
        
        r1_act = int(df_sub["bin_run1"].sum())
        r2_act = int(df_sub["bin_run2"].sum())
        r3_act = int(df_sub["bin_run3"].sum())
        tot_act = r1_act + r2_act + r3_act
        
        counts[sig] = tot_act
        counts_run1[sig] = r1_act
        counts_run2[sig] = r2_act
        counts_run3[sig] = r3_act
        
    total_activations_all_signals = sum(counts.values())
    
    records = []
    for sig in ALL_SIGNALS:
        tot = counts[sig]
        pct = (tot / total_activations_all_signals * 100.0) if total_activations_all_signals > 0 else 0.0
        records.append({
            "signal": sig,
            "threshold_used": thresholds[sig],
            "activations_run1": counts_run1[sig],
            "activations_run2": counts_run2[sig],
            "activations_run3": counts_run3[sig],
            "total_activations": tot,
            "relative_percentage": pct,
            "methodology_note": "Somma attivazioni binarie (Run 1 + Run 2 + Run 3) su finestre sovrapposte all'intervallo."
        })
        
    df_prop = pd.DataFrame(records)
    active_only_df = df_prop[df_prop["total_activations"] > 0].copy()
    zero_signals_list = df_prop[df_prop["total_activations"] == 0]["signal"].tolist()
    
    return df_prop, active_only_df, zero_signals_list


## 15. Funzioni di Visualizzazione e Plotting

Generazione dei grafici ad alta risoluzione (PNG $\ge 150$ DPI):
1. **Pipeline Overview:** Andamento continuo di tutti i segnali $S_1 - S_7$ nel tempo (asse x in secondi) per ciascuna run.
2. **Consensus Overview:** Livello di consenso inter-run ($0-3$) per ciascun segnale lungo l'intero video.
3. **Query Plots:** Grafici a linee con soglia, grafici a barre del consenso e grafici a torta della proporzionalità.


In [ ]:
def _clean_plot_title(title: str) -> str:
    """Rimuove dal titolo l'eventuale prefisso racchiuso tra parentesi quadre."""
    return re.sub(r'^\s*\[.*?\]\s*', '', str(title))


# Contatore progressivo per la nomenclatura dei file grafici esportati.
PLOT_FILE_COUNTER = 0


def _next_plot_path(save_dir: Path, current_name: str) -> Path:
    """Restituisce il percorso del prossimo grafico con prefisso progressivo 'graficiN_'."""
    global PLOT_FILE_COUNTER
    PLOT_FILE_COUNTER += 1
    return save_dir / f"grafici{PLOT_FILE_COUNTER}_{current_name}"


def _window_ticks(df: pd.DataFrame):
    """Restituisce posizioni ed etichette discrete delle finestre temporali."""
    positions = np.arange(len(df))
    labels = df["windowid"].astype(int).tolist()
    return positions, labels


def plot_pipeline_overview(dict_signals_raw: dict, df_temporal: pd.DataFrame, thresholds: dict, save_dir: Path):
    """
    Genera un grafico multi-pannello a 3 subplot (uno per ciascuna Run)
    mostrando l'andamento dei segnali S1raw-S7raw per finestra temporale.
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False, sharey=True)

    x_positions, x_labels = _window_ticks(df_temporal)

    for run_idx, ax in enumerate(axes, start=1):
        run_col = f"run{run_idx}"
        for sig in ALL_SIGNALS:
            sig_df = dict_signals_raw[sig]
            ax.plot(
                x_positions,
                sig_df[run_col],
                marker='o',
                markersize=4,
                label=sig,
                color=SIGNAL_COLORS[sig],
                linewidth=1.8
            )

        ax.set_title(
            _clean_plot_title(f"Pipeline VLM - Run {run_idx} (Valori Raw nel Tempo)"),
            fontsize=12,
            fontweight='bold',
            pad=8
        )
        ax.set_ylabel("Punteggio Raw", fontsize=10)
        ax.set_ylim(-0.05, 1.05)
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.axhline(
            0.5,
            color='gray',
            linestyle=':',
            alpha=0.6,
            label='Soglia default (0.5)'
        )
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels)
        ax.set_xlabel("Finestra Temporale" if run_idx == 3 else "", fontsize=10, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.5)

    # Legenda unica globale sotto tutti i subplot.
    handles, labels = axes[0].get_legend_handles_labels()
    # Legenda globale immediatamente sotto l'etichetta "Finestra Temporale"
    # dell'ultimo subplot, mantenendo un margine minimo per evitare sovrapposizioni.
    fig.legend(
        handles,
        labels,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.16),
        bbox_transform=axes[-1].transAxes,
        ncol=8,
        fontsize=9,
        frameon=True,
        borderaxespad=0.0
    )

    fig.tight_layout(rect=[0, 0.055, 1, 1])
    plot_path = _next_plot_path(save_dir, "00_pipeline_all_signals_temporal_overview.png")
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Salvato grafico: {plot_path.name}")


def plot_consensus_overview_all_signals(dict_signals_raw: dict, df_temporal: pd.DataFrame, thresholds: dict, save_dir: Path):
    """
    Genera un grafico con il livello di consenso inter-run (0-3 run attive)
    per ciascun segnale S1raw-S7raw, usando il numero della finestra sull'asse X.
    """
    fig, axes = plt.subplots(
        len(ALL_SIGNALS),
        1,
        figsize=(14, 14),
        sharex=False,
        sharey=True
    )
    x_positions, x_labels = _window_ticks(df_temporal)

    for idx, sig in enumerate(ALL_SIGNALS):
        ax = axes[idx]
        sig_df = dict_signals_raw[sig]
        thresh = thresholds[sig]
        df_bin = binarize_and_compute_consensus(sig_df, thresh)

        ax.step(
            x_positions,
            df_bin["active_runs_count"],
            where='mid',
            color=SIGNAL_COLORS[sig],
            linewidth=2,
            label=f"{sig} (Soglia {thresh})"
        )
        ax.fill_between(
            x_positions,
            df_bin["active_runs_count"],
            step='mid',
            alpha=0.25,
            color=SIGNAL_COLORS[sig]
        )
        ax.axhline(
            2,
            color='red',
            linestyle='--',
            linewidth=1.2,
            label='Maggioranza (2/3)'
        )

        ax.set_ylabel(f"{sig}\nRun Attive", fontsize=9, fontweight='bold')
        ax.set_yticks([0, 1, 2, 3])
        ax.set_ylim(-0.2, 3.2)
        ax.set_xticks(x_positions)
        ax.set_xticklabels(x_labels)
        ax.set_xlabel("Finestra Temporale" if idx == len(ALL_SIGNALS) - 1 else "", fontsize=9, fontweight='bold')
        ax.grid(True, linestyle=':', alpha=0.6)

    axes[0].set_title(
        _clean_plot_title("Consenso Inter-Run (0 - 3 Run Attive) per Segnale nel Tempo"),
        fontsize=13,
        fontweight='bold',
        pad=10
    )

    # Legenda unica globale sotto tutti i subplot, con tutte le indicazioni
    # in ordine S1raw-S7raw + Maggioranza (2/3).
    legend_handles = []
    legend_labels = []
    for idx, sig in enumerate(ALL_SIGNALS):
        handles_i, labels_i = axes[idx].get_legend_handles_labels()
        for h, lab in zip(handles_i, labels_i):
            if lab not in legend_labels and lab != 'Maggioranza (2/3)':
                legend_handles.append(h)
                legend_labels.append(lab)

    # Aggiunge una sola volta la linea della maggioranza.
    for idx in range(len(ALL_SIGNALS)):
        handles_i, labels_i = axes[idx].get_legend_handles_labels()
        for h, lab in zip(handles_i, labels_i):
            if lab == 'Maggioranza (2/3)':
                legend_handles.append(h)
                legend_labels.append(lab)
                break
        if 'Maggioranza (2/3)' in legend_labels:
            break

    # Legenda globale immediatamente sotto l'etichetta "Finestra Temporale"
    # dell'ultimo subplot, mantenendo un margine minimo per evitare sovrapposizioni.
    fig.legend(
        legend_handles,
        legend_labels,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.16),
        bbox_transform=axes[-1].transAxes,
        ncol=8,
        fontsize=8,
        frameon=True,
        borderaxespad=0.0
    )

    fig.tight_layout(rect=[0, 0.055, 1, 1])
    plot_path = _next_plot_path(save_dir, "00_consensus_overview_all_signals.png")
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Salvato grafico: {plot_path.name}")


def plot_global_absence_query(res: dict, save_dir: Path):
    """
    Genera i grafici per una query di assenza globale:
    1. Andamento del segnale nelle 3 run con linea di soglia.
    2. Conteggio attivazioni per run, mostrato solo se esiste almeno un'attivazione.
    """
    q_id = res["query_id"]
    sig = res["signal"]
    thresh = res["threshold"]
    df = res["window_details"]

    acts = [
        res["summary"]["activations_run1"],
        res["summary"]["activations_run2"],
        res["summary"]["activations_run3"]
    ]
    has_any_activation = any(a > 0 for a in acts)

    if has_any_activation:
        fig, (ax1, ax2) = plt.subplots(
            1,
            2,
            figsize=(14, 4.5),
            gridspec_kw={'width_ratios': [2.2, 1]},
            sharex=False
        )
        axes = [ax1, ax2]
    else:
        fig, ax1 = plt.subplots(1, 1, figsize=(10, 4.5))
        axes = [ax1]

    x_positions, x_labels = _window_ticks(df)

    ax1.plot(
        x_positions,
        df["run1"],
        marker='o',
        label='Run 1',
        color=RUN_COLORS[1],
        linewidth=1.5
    )
    ax1.plot(
        x_positions,
        df["run2"],
        marker='s',
        label='Run 2',
        color=RUN_COLORS[2],
        linewidth=1.5
    )
    ax1.plot(
        x_positions,
        df["run3"],
        marker='^',
        label='Run 3',
        color=RUN_COLORS[3],
        linewidth=1.5
    )
    ax1.axhline(
        thresh,
        color='red',
        linestyle='--',
        linewidth=1.5,
        label=f'Soglia ({thresh})'
    )

    ax1.set_title(
        _clean_plot_title(f"[{q_id}] {sig} - Andamento nelle 3 Run"),
        fontsize=11,
        fontweight='bold'
    )
    ax1.set_xlabel("", fontsize=10, fontweight='bold')
    ax1.set_ylabel("Punteggio Raw", fontsize=10)
    ax1.set_ylim(-0.05, 1.05)
    ax1.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(x_labels)
    ax1.grid(True, linestyle='--', alpha=0.5)

    if has_any_activation:
        # Barplot conteggi
        runs = ['Run 1', 'Run 2', 'Run 3']
        bars = ax2.bar(
            runs,
            acts,
            color=[RUN_COLORS[1], RUN_COLORS[2], RUN_COLORS[3]],
            width=0.5,
            edgecolor='black'
        )
        ax2.set_title(
            _clean_plot_title(f"Conteggio Attivazioni (>= {thresh})"),
            fontsize=11,
            fontweight='bold'
        )
        ax2.set_ylabel("N. Finestre Attive", fontsize=10)
        ax2.set_xlabel("Run", fontsize=10, fontweight='bold')
        ax2.set_ylim(0, max(max(acts) + 1, 4))
        ax2.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        ax2.grid(axis='y', linestyle='--', alpha=0.5)

        for bar in bars:
            height = bar.get_height()
            ax2.annotate(
                f'{int(height)}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center',
                va='bottom',
                fontweight='bold'
            )

        # Legenda unica globale a destra del grafico, senza sovrapporre l'area tracciata.
        handles, labels = ax1.get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(0.82, 0.5),
            ncol=1,
            fontsize=9,
            frameon=True,
            borderaxespad=0.0
        )
        fig.tight_layout(rect=[0, 0, 0.78, 1])
    else:
        # Anche senza conteggio attivazioni, la legenda resta esterna a destra.
        handles, labels = ax1.get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc='center left',
            bbox_to_anchor=(0.82, 0.5),
            ncol=1,
            fontsize=9,
            frameon=True,
            borderaxespad=0.0
        )
        fig.tight_layout(rect=[0, 0, 0.78, 1])

    plot_path = _next_plot_path(save_dir, f"{q_id}_global_absence.png")
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Salvato grafico: {plot_path.name}")


def plot_interval_query(res: dict, dict_signals_raw: dict, thresholds: dict, save_dir: Path):
    """
    Genera i grafici per una query di intervallo:
    1. Linee raw delle 3 run con soglia.
    2. Barre del consenso inter-run (0-3).
    3. Grafico a torta di proporzionalità S1raw-S7raw nell'intervallo.
    """
    q_id = res["query_id"]
    sig = res["signal"]
    time_range = res["time_range"]
    thresh = res["threshold"]
    df_win = res["window_details"]

    # 1. Grafico temporale locale + consenso
    fig, (ax1, ax2) = plt.subplots(
        2,
        1,
        figsize=(10, 7),
        sharex=False
    )

    if not df_win.empty:
        x_positions, x_labels = _window_ticks(df_win)

        ax1.plot(
            x_positions,
            df_win["run1"],
            marker='o',
            label='Run 1',
            color=RUN_COLORS[1],
            linewidth=1.8
        )
        ax1.plot(
            x_positions,
            df_win["run2"],
            marker='s',
            label='Run 2',
            color=RUN_COLORS[2],
            linewidth=1.8
        )
        ax1.plot(
            x_positions,
            df_win["run3"],
            marker='^',
            label='Run 3',
            color=RUN_COLORS[3],
            linewidth=1.8
        )
        ax1.axhline(
            thresh,
            color='red',
            linestyle='--',
            linewidth=1.5,
            label=f'Soglia ({thresh})'
        )
        ax1.set_title(
            _clean_plot_title(f"[{q_id}] {sig} - Valori Raw nell'Intervallo"),
            fontsize=11,
            fontweight='bold'
        )
        ax1.set_ylabel("Punteggio Raw", fontsize=10)
        ax1.set_ylim(-0.05, 1.05)
        ax1.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax1.set_xticks(x_positions)
        ax1.set_xticklabels(x_labels)
        ax1.set_xlabel("", fontsize=10, fontweight='bold')
        ax1.grid(True, linestyle='--', alpha=0.5)

        # Barre consenso
        bar_colors = [
            '#d95f02' if c >= 2 else '#7570b3'
            for c in df_win["active_runs_count"]
        ]
        bars = ax2.bar(
            x_positions,
            df_win["active_runs_count"],
            color=bar_colors,
            width=0.5,
            edgecolor='black'
        )
        ax2.axhline(
            2,
            color='red',
            linestyle='--',
            linewidth=1.2,
            label='Soglia Maggioranza (>=2)'
        )
        ax2.set_title(
            _clean_plot_title("Consenso Inter-Run"),
            fontsize=11,
            fontweight='bold'
        )
        ax2.set_ylabel("Run Attive (0-3)", fontsize=10)
        ax2.set_ylim(0, 3.4)
        ax2.set_yticks([0, 1, 2, 3])
        ax2.set_xticks(x_positions)
        ax2.set_xticklabels(x_labels)
        ax2.set_xlabel("Finestra Temporale", fontsize=10, fontweight='bold')
        ax2.grid(axis='y', linestyle='--', alpha=0.5)

        for bar in bars:
            h = bar.get_height()
            ax2.annotate(
                f'{int(h)}/3',
                xy=(bar.get_x() + bar.get_width() / 2, h),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center',
                va='bottom',
                fontsize=9,
                fontweight='bold'
            )

        # Legenda unica globale sotto entrambi i subplot.
        handles1, labels1 = ax1.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        # Legenda globale immediatamente sotto l'etichetta "Finestra Temporale"
        # dell'ultimo subplot.
        fig.legend(
            handles1 + handles2,
            labels1 + labels2,
            loc='upper center',
            bbox_to_anchor=(0.5, -0.16),
            bbox_transform=ax2.transAxes,
            ncol=5,
            fontsize=8,
            frameon=True,
            borderaxespad=0.0
        )
        fig.tight_layout(rect=[0, 0.055, 1, 1])
    else:
        ax1.text(0.5, 0.5, "Nessuna finestra inclusa", ha='center', va='center')
        ax2.text(0.5, 0.5, "Nessuna finestra inclusa", ha='center', va='center')
        ax1.set_xlabel("Finestra Temporale", fontsize=10, fontweight='bold')
        ax2.set_xlabel("Finestra Temporale", fontsize=10, fontweight='bold')
        fig.tight_layout()

    plot_path_times = _next_plot_path(
        save_dir,
        f"{q_id}_temporal_and_consensus.png"
    )
    fig.savefig(plot_path_times, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Salvato grafico: {plot_path_times.name}")

    # 2. Grafico a Torta di Proporzionalità
    df_prop, active_only, zero_sigs = compute_interval_proportionality(
        time_range,
        dict_signals_raw,
        thresholds
    )

    fig, ax = plt.subplots(figsize=(9.5, 6.5))

    if not active_only.empty:
        colors_pie = [SIGNAL_COLORS[s] for s in active_only["signal"]]
        wedges, texts, autotexts = ax.pie(
            active_only["total_activations"],
            labels=active_only["signal"],
            autopct='%1.1f%%',
            startangle=140,
            colors=colors_pie,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
            textprops={'fontsize': 10, 'fontweight': 'bold'},
            labeldistance=1.08,
            pctdistance=0.68
        )

        for at in autotexts:
            at.set_color('white')
            at.set_fontsize(10)

        zero_text = (
            f"Segnali con 0 attivazioni: {', '.join(zero_sigs)}"
            if zero_sigs
            else "Tutti i segnali hanno >= 1 attivazione."
        )

        ax.set_title(
            _clean_plot_title(
                f"[{q_id}] Proporzionalità Attivazioni S1raw-S7raw"
            ),
            fontsize=11,
            fontweight='bold',
            pad=12
        )
        fig.text(
            0.5,
            0.02,
            zero_text,
            ha='center',
            fontsize=9,
            style='italic',
            color='#555555'
        )

        # Legenda esterna a destra, senza coprire etichette o percentuali.
        ax.legend(
            wedges,
            [
                f"{s} ({int(a)} att.)"
                for s, a in zip(
                    active_only["signal"],
                    active_only["total_activations"]
                )
            ],
            title="Segnali Attivi",
            loc="center left",
            bbox_to_anchor=(1.05, 0.5),
            fontsize=9
        )
        fig.tight_layout(rect=[0, 0.06, 0.78, 1])
    else:
        ax.text(
            0.5,
            0.5,
            "Nessuna attivazione rilevata per alcun segnale\nnell'intervallo",
            ha='center',
            va='center',
            fontsize=11,
            fontweight='bold',
            color='red'
        )
        ax.set_title(
            _clean_plot_title(
                f"[{q_id}] Proporzionalità S1raw-S7raw"
            ),
            fontsize=11,
            fontweight='bold'
        )
        fig.tight_layout()

    plot_path_pie = _next_plot_path(
        save_dir,
        f"{q_id}_proportionality_pie.png"
    )
    fig.savefig(plot_path_pie, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"Salvato grafico: {plot_path_pie.name}")


## 16. Funzioni di Esportazione Report Excel

Genera il file Excel finale multi-foglio `risultati_analisi_sn_raw.xlsx` contenente i 6 fogli standardizzati e formattati con stili professionali (grassetto, autofilter, blocco riquadri, larghezza colonne ottimizzata, percentuali formattate):
1. `metadata`
2. `global_absence_summary`
3. `interval_summary`
4. `window_level_details`
5. `proportionality_details`
6. `all_signals_consensus`


In [ ]:
def export_excel_report(output_file: Path,
                        metadata_records: list,
                        global_absence_records: list,
                        interval_summary_records: list,
                        window_level_records: list,
                        proportionality_records: list,
                        all_signals_consensus_records: list):
    """
    Esporta tutti i risultati dell'analisi in un workbook Excel professionale a 6 fogli.
    """
    wb = openpyxl.Workbook()
    # Rimuove il foglio di default
    wb.remove(wb.active)
    
    # Stili standard
    header_fill = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    data_font = Font(name="Calibri", size=10)
    border_thin = Border(left=Side(style='thin', color='D9D9D9'),
                         right=Side(style='thin', color='D9D9D9'),
                         top=Side(style='thin', color='D9D9D9'),
                         bottom=Side(style='thin', color='D9D9D9'))
    
    def format_sheet(ws, title):
        ws.views.sheetView[0].showGridLines = True
        ws.freeze_panes = "A2"
        # Header formatting
        for col_idx in range(1, ws.max_column + 1):
            cell = ws.cell(row=1, column=col_idx)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            
        # Data formatting & column width auto-fit
        for col in ws.columns:
            max_len = 0
            col_letter = get_column_letter(col[0].column)
            for cell in col:
                cell.border = border_thin
                if cell.row > 1:
                    cell.font = data_font
                    # Format numbers
                    if isinstance(cell.value, float):
                        if "pct" in str(ws.cell(row=1, column=cell.column).value).lower() or "percentage" in str(ws.cell(row=1, column=cell.column).value).lower():
                            cell.number_format = '0.0%'
                            cell.value = cell.value / 100.0 if cell.value > 1.0 else cell.value
                        else:
                            cell.number_format = '0.0000'
                    elif isinstance(cell.value, int):
                        cell.number_format = '#,##0'
                val_str = str(cell.value or '')
                max_len = max(max_len, len(val_str))
            ws.column_dimensions[col_letter].width = max(max_len + 4, 12)
            
    # 1. Metadata
    ws_meta = wb.create_sheet(title="metadata")
    df_meta = pd.DataFrame(metadata_records)
    for r_idx, row in enumerate(df_meta.itertuples(index=False), start=1):
        if r_idx == 1:
            ws_meta.append(list(df_meta.columns))
        ws_meta.append(list(row))
    format_sheet(ws_meta, "metadata")
    
    # 2. Global Absence Summary
    ws_gas = wb.create_sheet(title="global_absence_summary")
    df_gas = pd.DataFrame(global_absence_records) if global_absence_records else pd.DataFrame(columns=["Info"])
    for r_idx, row in enumerate(df_gas.itertuples(index=False), start=1):
        if r_idx == 1:
            ws_gas.append(list(df_gas.columns))
        ws_gas.append(list(row))
    format_sheet(ws_gas, "global_absence_summary")
    
    # 3. Interval Summary
    ws_int = wb.create_sheet(title="interval_summary")
    df_int = pd.DataFrame(interval_summary_records) if interval_summary_records else pd.DataFrame(columns=["Info"])
    for r_idx, row in enumerate(df_int.itertuples(index=False), start=1):
        if r_idx == 1:
            ws_int.append(list(df_int.columns))
        ws_int.append(list(row))
    format_sheet(ws_int, "interval_summary")
    
    # 4. Window Level Details
    ws_win = wb.create_sheet(title="window_level_details")
    df_win = pd.DataFrame(window_level_records) if window_level_records else pd.DataFrame(columns=["Info"])
    for r_idx, row in enumerate(df_win.itertuples(index=False), start=1):
        if r_idx == 1:
            ws_win.append(list(df_win.columns))
        ws_win.append(list(row))
    format_sheet(ws_win, "window_level_details")
    
    # 5. Proportionality Details
    ws_prop = wb.create_sheet(title="proportionality_details")
    df_prop = pd.DataFrame(proportionality_records) if proportionality_records else pd.DataFrame(columns=["Info"])
    for r_idx, row in enumerate(df_prop.itertuples(index=False), start=1):
        if r_idx == 1:
            ws_prop.append(list(df_prop.columns))
        ws_prop.append(list(row))
    format_sheet(ws_prop, "proportionality_details")
    
    # 6. All Signals Consensus
    ws_all = wb.create_sheet(title="all_signals_consensus")
    df_all = pd.DataFrame(all_signals_consensus_records) if all_signals_consensus_records else pd.DataFrame(columns=["Info"])
    for r_idx, row in enumerate(df_all.itertuples(index=False), start=1):
        if r_idx == 1:
            ws_all.append(list(df_all.columns))
        ws_all.append(list(row))
    format_sheet(ws_all, "all_signals_consensus")
    
    wb.save(output_file)
    print(f"Report Excel salvato con successo: {output_file.resolve()}")


## 17. Esecuzione della Pipeline di Analisi

Caricamento dei file (o generazione di dataset di test se i file non sono ancora presenti nella directory di lavoro), validazione, esecuzione di tutte le query configurate, generazione grafici ed export Excel.


In [ ]:
# ==============================================================================
# ESECUZIONE COMPLETA DELLA PIPELINE
# ==============================================================================

# 1. Verifica ed eventuale generazione dati di test se i file non esistono
def ensure_sample_files_exist(file_list: list):
    """Se i file Excel richiesti non sono presenti, genera dataset sintetici realistici per test immediato."""
    missing = [f for f in file_list if not Path(f).exists()]
    if missing:
        print(f"Attenzione: {len(missing)} file di input non trovati. Generazione automatica di file sintetici realistici per consentire l'esecuzione immediata...")
        # Generazione 18 finestre da 2 secondi ciascuna (0 - 36 secondi)
        np.random.seed(42)
        n_windows = 18
        win_dur = 2.0
        starttimes = np.arange(0, n_windows * win_dur, win_dur)
        endtimes = starttimes + win_dur
        windowids = np.arange(1, n_windows + 1)
        
        for idx, fname in enumerate(file_list, start=1):
            if not Path(fname).exists():
                data = {
                    "windowid": windowids,
                    "starttimes": starttimes,
                    "endtimes": endtimes,
                    "windowdurations": [win_dur] * n_windows,
                    # S1: attivo nella prima metà
                    "S1raw": np.clip(np.where(starttimes <= 16, 0.75 + np.random.normal(0, 0.1, n_windows), 0.15 + np.random.normal(0, 0.05, n_windows)), 0, 1),
                    # S2: basso
                    "S2raw": np.clip(0.1 + np.random.normal(0, 0.05, n_windows), 0, 1),
                    # S3: attivo intorno a 26s
                    "S3raw": np.clip(np.where((starttimes >= 24) & (starttimes <= 28), 0.85 + np.random.normal(0, 0.08, n_windows), 0.08 + np.random.normal(0, 0.03, n_windows)), 0, 1),
                    # S4: attivo tra 16s e 28s
                    "S4raw": np.clip(np.where((starttimes >= 16) & (starttimes <= 28), 0.80 + np.random.normal(0, 0.09, n_windows), 0.10 + np.random.normal(0, 0.04, n_windows)), 0, 1),
                    # S5: attivo tra 28s e 36s
                    "S5raw": np.clip(np.where(starttimes >= 28, 0.88 + np.random.normal(0, 0.06, n_windows), 0.05 + np.random.normal(0, 0.02, n_windows)), 0, 1),
                    # S6: assente globale (sempre < 0.5)
                    "S6raw": np.clip(0.04 + np.random.normal(0, 0.02, n_windows), 0, 0.25),
                    # S7: assente globale (sempre < 0.5)
                    "S7raw": np.clip(0.02 + np.random.normal(0, 0.01, n_windows), 0, 0.20),
                }
                df_synth = pd.DataFrame(data)
                with pd.ExcelWriter(fname, engine='openpyxl') as writer:
                    df_synth.to_excel(writer, sheet_name=SHEET_NAME, index=False)
                print(f"  -> Creato file di test: {fname}")

ensure_sample_files_exist(INPUT_FILES)

# 2. Lettura e validazione delle 3 run
print("\n--- FASE 1: Lettura e validazione dei file di input ---")
run_dfs = []
for fpath in INPUT_FILES:
    df_r = load_and_validate_run_file(fpath, sheet_name=SHEET_NAME, mandatory_cols=MANDATORY_COLS)
    run_dfs.append(df_r)
    print(f"File caricato con successo: {fpath} ({len(df_r)} finestre)")

# 3. Allineamento
print("\n--- FASE 2: Allineamento temporale delle run ---")
df_temporal, dict_signals_raw = align_runs(run_dfs)

# ------------------------------------------------------------------------------
# ESCLUSIONE FINESTRE
# Le finestre indicate in EXCLUDED_WINDOWS vengono rimosse dal dataset analizzato
# prima della generazione dei grafici, delle query e di tutte le metriche.
# ------------------------------------------------------------------------------
EXCLUDED_WINDOWS = sorted(set(int(w) for w in EXCLUDED_WINDOWS))

available_window_ids = set(df_temporal["windowid"].astype(int).tolist())
unknown_excluded_windows = sorted(set(EXCLUDED_WINDOWS) - available_window_ids)

if unknown_excluded_windows:
    print(
        f"Attenzione: le seguenti finestre indicate in EXCLUDED_WINDOWS "
        f"non sono presenti nei dati e verranno ignorate: {unknown_excluded_windows}"
    )

if EXCLUDED_WINDOWS:
    exclusion_mask = ~df_temporal["windowid"].isin(EXCLUDED_WINDOWS)
    excluded_count = int((~exclusion_mask).sum())

    df_temporal = df_temporal.loc[exclusion_mask].copy().reset_index(drop=True)

    for sig in dict_signals_raw:
        dict_signals_raw[sig] = (
            dict_signals_raw[sig]
            .loc[exclusion_mask]
            .copy()
            .reset_index(drop=True)
        )

    print(
        f"Esclusione finestre applicata: {excluded_count} finestra/e escluse "
        f"dal calcolo delle metriche: {EXCLUDED_WINDOWS}"
    )
else:
    excluded_count = 0
    print("Nessuna finestra esclusa: EXCLUDED_WINDOWS = []")

total_windows_original = total_windows = len(df_temporal)

if total_windows == 0:
    raise ValueError(
        "Tutte le finestre sono state escluse. "
        "Controllare EXCLUDED_WINDOWS prima di proseguire."
    )

video_duration = float(df_temporal["endtimes"].max())
print(f"Allineamento completato: {total_windows} finestre, durata video rilevata: {video_duration:.2f} secondi.")

# 4. Generazione grafici overview
print("\n--- FASE 3: Generazione grafici globali di overview ---")
plot_pipeline_overview(dict_signals_raw, df_temporal, SN_THRESHOLDS, PLOTS_DIR)
plot_consensus_overview_all_signals(dict_signals_raw, df_temporal, SN_THRESHOLDS, PLOTS_DIR)

# 5. Esecuzione query
print("\n--- FASE 4: Esecuzione delle query analitiche ---")
global_absence_records = []
interval_summary_records = []
window_level_records = []
proportionality_records = []
query_results = []

for q in QUERIES:
    q_type = q["type"]
    q_id = q["id"]
    sig = q["signal"]
    
    if q_type == "global_absence":
        res = analyze_global_absence(q, dict_signals_raw, SN_THRESHOLDS)
        query_results.append(res)
        global_absence_records.append(res["summary"])
        plot_global_absence_query(res, PLOTS_DIR)
        
        # Dettaglio finestre per foglio window_level_details
        df_w = res["window_details"]
        for _, row in df_w.iterrows():
            window_level_records.append({
                "query_id": q_id,
                "signal": sig,
                "windowid": int(row["windowid"]),
                "starttimes": float(row["starttimes"]),
                "endtimes": float(row["endtimes"]),
                "run1_raw": float(row["run1"]),
                "run2_raw": float(row["run2"]),
                "run3_raw": float(row["run3"]),
                "bin_run1": int(row["bin_run1"]),
                "bin_run2": int(row["bin_run2"]),
                "bin_run3": int(row["bin_run3"]),
                "active_runs_count": int(row["active_runs_count"]),
                "consensus_str": row["consensus_str"],
                "majority_consensus": "SÌ" if row["majority_consensus"] else "NO",
                "complete_consensus": "SÌ" if row["complete_consensus"] else "NO"
            })
            
    elif q_type in ["single_signal_interval", "inter_run_consensus"]:
        res = analyze_single_signal_interval(q, dict_signals_raw, SN_THRESHOLDS)
        query_results.append(res)
        interval_summary_records.append(res["summary"])
        plot_interval_query(res, dict_signals_raw, SN_THRESHOLDS, PLOTS_DIR)
        
        # Dettaglio finestre per foglio window_level_details
        df_w = res["window_details"]
        for _, row in df_w.iterrows():
            window_level_records.append({
                "query_id": q_id,
                "signal": sig,
                "windowid": int(row["windowid"]),
                "starttimes": float(row["starttimes"]),
                "endtimes": float(row["endtimes"]),
                "run1_raw": float(row["run1"]),
                "run2_raw": float(row["run2"]),
                "run3_raw": float(row["run3"]),
                "bin_run1": int(row["bin_run1"]),
                "bin_run2": int(row["bin_run2"]),
                "bin_run3": int(row["bin_run3"]),
                "active_runs_count": int(row["active_runs_count"]),
                "consensus_str": row["consensus_str"],
                "majority_consensus": "SÌ" if row["majority_consensus"] else "NO",
                "complete_consensus": "SÌ" if row["complete_consensus"] else "NO"
            })
            
        # Calcolo proporzionalità per foglio proportionality_details
        df_prop, _, _ = compute_interval_proportionality(q["time_range_s"], dict_signals_raw, SN_THRESHOLDS)
        for _, p_row in df_prop.iterrows():
            proportionality_records.append({
                "query_id": q_id,
                "interval": f"[{q['time_range_s'][0]}s, {q['time_range_s'][1]}s]",
                "signal": p_row["signal"],
                "threshold_used": p_row["threshold_used"],
                "activations_run1": p_row["activations_run1"],
                "activations_run2": p_row["activations_run2"],
                "activations_run3": p_row["activations_run3"],
                "total_activations": p_row["total_activations"],
                "relative_percentage": p_row["relative_percentage"],
                "definition_note": p_row["methodology_note"]
            })

# 6. Tabella all_signals_consensus per audit
print("\n--- FASE 5: Costruzione audit completo all_signals_consensus ---")
all_signals_consensus_records = []
for sig in ALL_SIGNALS:
    sig_df = dict_signals_raw[sig]
    thresh = SN_THRESHOLDS[sig]
    df_bin = binarize_and_compute_consensus(sig_df, thresh)
    for _, row in df_bin.iterrows():
        all_signals_consensus_records.append({
            "windowid": int(row["windowid"]),
            "starttimes": float(row["starttimes"]),
            "endtimes": float(row["endtimes"]),
            "signal": sig,
            "threshold": thresh,
            "run1_raw": float(row["run1"]),
            "run2_raw": float(row["run2"]),
            "run3_raw": float(row["run3"]),
            "bin_run1": int(row["bin_run1"]),
            "bin_run2": int(row["bin_run2"]),
            "bin_run3": int(row["bin_run3"]),
            "active_runs_count": int(row["active_runs_count"]),
            "consensus_str": row["consensus_str"],
            "majority_consensus": "SÌ" if row["majority_consensus"] else "NO",
            "complete_consensus": "SÌ" if row["complete_consensus"] else "NO"
        })

# 7. Metadata
metadata_records = [
    {"Parametro": "Timestamp Analisi", "Valore": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")},
    {"Parametro": "File Run 1", "Valore": str(Path(INPUT_FILES[0]).name)},
    {"Parametro": "File Run 2", "Valore": str(Path(INPUT_FILES[1]).name)},
    {"Parametro": "File Run 3", "Valore": str(Path(INPUT_FILES[2]).name)},
    {"Parametro": "Foglio Excel Letto", "Valore": SHEET_NAME},
    {"Parametro": "Numero Totale Run", "Valore": 3},
    {"Parametro": "Numero Finestre Temporali Analizzate", "Valore": total_windows},
    {"Parametro": "Numero Finestre Temporali Originali", "Valore": total_windows_original},
    {"Parametro": "Finestre Escluse", "Valore": str(EXCLUDED_WINDOWS) if EXCLUDED_WINDOWS else "Nessuna"},
    {"Parametro": "Numero Finestre Escluse", "Valore": excluded_count},
    {"Parametro": "Durata Totale Video Rilevata", "Valore": f"{video_duration:.2f} secondi"},
    {"Parametro": "Segnali Analizzati", "Valore": ", ".join(ALL_SIGNALS)},
    {"Parametro": "Soglie Applicate", "Valore": str(SN_THRESHOLDS)},
    {"Parametro": "Regola Consenso Maggioritario", "Valore": "Almeno 2 run attive su 3 (>= 2/3)"},
    {"Parametro": "Regola Consenso Completo", "Valore": "3 run attive su 3 (== 3/3)"},
    {"Parametro": "Metodo Selezione Finestre", "Valore": "max(starttimes, t_start) < min(endtimes, t_end)"},
    {"Parametro": "Numero Query Eseguite", "Valore": len(QUERIES)},
    {"Parametro": "Definizione Proporzionalità", "Valore": "Somma attivazioni binarie totali (Run 1+2+3) nell'intervallo"}
]

# 8. Esportazione Report Excel
print("\n--- FASE 6: Esportazione report Excel a 6 fogli ---")
export_excel_report(
    EXCEL_OUTPUT_PATH,
    metadata_records,
    global_absence_records,
    interval_summary_records,
    window_level_records,
    proportionality_records,
    all_signals_consensus_records
)


## 18. Stampa dei Risultati Principali

Riepilogo tabellare a schermo dei risultati ottenuti da ciascuna query configurata.


In [ ]:
print("=" * 80)
print("RIASSUNTO ESECUTIVO DEI RISULTATI DELLE QUERY")
print("=" * 80)

for res in query_results:
    q_id = res["query_id"]
    q_type = res["query_type"]
    sig = res["signal"]
    
    if q_type == "global_absence":
        s = res["summary"]
        print(f"\n[QUERY ASSENZA GLOBALE] -> ID: {q_id} | Segnale: {sig}")
        print(f"  - Assenza Totale Osservata (< {s['threshold']}): {s['always_below_threshold']}")
        print(f"  - Massimo Globale: {s['max_global']:.4f} (R1: {s['max_run1']:.4f}, R2: {s['max_run2']:.4f}, R3: {s['max_run3']:.4f})")
        print(f"  - Attivazioni Totali: {s['activations_total']} (R1: {s['activations_run1']}, R2: {s['activations_run2']}, R3: {s['activations_run3']})")
        print(f"  - Consenso Finestre: [0/3: {s['consensus_0_3_count']}, 1/3: {s['consensus_1_3_count']}, 2/3: {s['consensus_2_3_count']}, 3/3: {s['consensus_3_3_count']}]")
        print(f"  - Nota: {s['descriptive_text']}")
        
    elif q_type in ["single_signal_interval", "inter_run_consensus"]:
        s = res["summary"]
        print(f"\n[QUERY INTERVALLO LOCALE] -> ID: {q_id} | Segnale: {sig} | Intervallo: {s['time_range_str']}")
        print(f"  - Finestre Incluse ({s['num_windows']}): ID [{s['included_windows_ids']}]")
        print(f"  - Copertura Consenso Maggioritario (>=2/3): {s['majority_consensus_coverage_pct']:.1f}% ({s['majority_consensus_windows']}/{s['num_windows']} finestre)")
        print(f"  - Copertura Consenso Completo (3/3): {s['complete_consensus_coverage_pct']:.1f}% ({s['complete_consensus_windows']}/{s['num_windows']} finestre)")
        print(f"  - Attivazioni per Run: Run1={s['activations_run1']}, Run2={s['activations_run2']}, Run3={s['activations_run3']} (Max: {s['max_observed_value']:.4f})")
        print(f"  - Nota: {s['descriptive_notes']}")

print("\n" + "=" * 80)


## 19. Verifica Integrità degli Output Prodotti

Controllo deterministico dell'esistenza del file Excel e di tutti i grafici generati nella cartella `plots`.


In [ ]:
print("--- VERIFICA DEGLI ARTEFATTI PRODOTTI ---")

# 1. Verifica Excel
if EXCEL_OUTPUT_PATH.exists():
    excel_size_kb = EXCEL_OUTPUT_PATH.stat().st_size / 1024.0
    print(f"[OK] File Excel generato: {EXCEL_OUTPUT_PATH.resolve()} ({excel_size_kb:.1f} KB)")
    
    # Verifica fogli presenti
    wb_check = openpyxl.load_workbook(EXCEL_OUTPUT_PATH, read_only=True)
    print(f"     Fogli presenti ({len(wb_check.sheetnames)}): {wb_check.sheetnames}")
else:
    print(f"[ERRORE] File Excel non trovato in {EXCEL_OUTPUT_PATH.resolve()}")

# 2. Verifica Grafici
generated_plots = sorted(list(PLOTS_DIR.glob("*.png")))
print(f"\n[OK] Totale grafici PNG ad alta risoluzione generati ({len(generated_plots)}):")
for p in generated_plots:
    p_size_kb = p.stat().st_size / 1024.0
    print(f"  - {p.name} ({p_size_kb:.1f} KB)")


## 20. Guida Operativa per l'Analisi di Nuovi Video

Per applicare questa pipeline di analisi a un nuovo video industriale o a nuovi output VLM:

1. **Aggiornare i File di Input (Sezione 3):**
   Modificare la lista `INPUT_FILES` inserendo i percorsi dei 3 file Excel del nuovo video:
   ```python
   INPUT_FILES = [
       "nuovo_video_run1.xlsx",
       "nuovo_video_run2.xlsx",
       "nuovo_video_run3.xlsx",
   ]
   ```
2. **Calibrare le Soglie (Sezione 4):**
   Modificare il dizionario `SN_THRESHOLDS` se si desiderano soglie di attivazione diverse per specifici segnali.
3. **Configurare le Query (Sezione 5):**
   Modificare o aggiungere query nella lista `QUERIES` specificando gli intervalli temporali `[inizio_s, fine_s]`, i segnali `S1raw - S7raw` e la tipologia di analisi desiderata (`global_absence`, `single_signal_interval`, `inter_run_consensus`).
4. **Eseguire il Notebook:**
   Eseguire tutte le celle (*Kernel -> Restart & Run All*). Tutti i report Excel e i grafici PNG verranno aggiornati automaticamente nella cartella `output_sn_raw_analysis`.
